# Embedding Model Comparison Benchmark - EmbedComp
**Models:** e5-base-v2 · bge-base-en-v1.5 · multilingual-e5-base · all-MiniLM-L6-v2  
**Dataset:** BEIR trec-covid corpus  
**Metrics:** Encode throughput · Query latency (mean/p95/p99) · Recall@K · MRR · Cosine distribution

#### Search/Recommendation System Metrics Overview

| Metric | What it Measures | Ideal Improvement / Interpretation |
| :--- | :--- | :--- |
| **Encode throughput** | How many documents the model can embed per second (system capacity). | ⬆️ Higher (Higher is better) |
| **Query latency (mean)** | Average time from "user types query" to "results returned," in milliseconds. | ⬇️ Lower (Lower is better) |
| **Query latency (p95)** | The time by which 95% of all queries finish—a realistic worst-case for most users. | ⬇️ Lower (Lower is better) |
| **Query latency (p99)** | The time by which 99% of all queries finish—representing tail latency experienced by the slowest 1 in 100 users. | ⬇️ Lower (Lower is better) |
| **Recall@K** *(e.g., Recall@3)* | Measures if a relevant document appears within the top $K$ results. ($\text{ideal} = 1.0$) | ⬆️ Higher (Higher is better; ideal = 1.0) |
| **MRR** | Mean Reciprocal Rank. How high the relevant document ranks on average (e.g., if always first, rank is 1.0). | ⬆️ Higher (Closer to 1.0 is better) |
| **Cosine distribution** | The spread of similarity scores across top-K hits. Measures how distinct your best results are from each other. | 📈 Mean higher · Spread narrower (Ideal is a high mean with minimal variance) |

In [ ]:
# Run once to install deps
!pip install sentence-transformers datasets faiss-cpu plotly pandas numpy

In [2]:
import time
import numpy as np
import pandas as pd
import faiss
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from sentence_transformers import SentenceTransformer
from datasets import load_dataset
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')
print('All imports OK')

All imports OK


In [19]:
# ── CONFIG ── swap models / queries freely here ──────────────────────
MODELS = {
    'e5-base-v2':           'intfloat/e5-base-v2',
    'bge-base-en':          'BAAI/bge-base-en-v1.5',
    'multilingual-e5-base': 'intfloat/multilingual-e5-base',
    'all-MiniLM-L6-v2':    'sentence-transformers/all-MiniLM-L6-v2',
    'nomic-embed-v1':       'nomic-ai/nomic-embed-text-v1'
}

QUERY_PREFIX = {
    'e5-base-v2':           'query: ',
    'bge-base-en':          'Represent this sentence for searching relevant passages: ',
    'multilingual-e5-base': 'query: ',
    'all-MiniLM-L6-v2':    '',
    'nomic-embed-v1':               'query: '
}
DOC_PREFIX = {
    'e5-base-v2':           'passage: ',
    'bge-base-en':          '',
    'multilingual-e5-base': 'passage: ',
    'all-MiniLM-L6-v2':    '',
    'nomic-embed-v1':       'passage: '
}

CORPUS_SIZE  = 500   # increase on GPU
BATCH_SIZE   = 16
TOP_K        = 5
LATENCY_RUNS = 10    # number of runs to average latency over

# Ground-truth: query -> list of relevant doc indices in corpus
# These will be computed dynamically using e5-base-v2 as reference
QUERIES = [
    'How does COVID-19 affect lung tissue?',
    'What are the symptoms of coronavirus infection?',
    'How is PCR testing used to detect COVID-19?',
    'What treatments exist for severe COVID cases?',
    'How does the spike protein enable viral entry?',
]
GROUND_TRUTH = {}  # Will be populated after loading corpus

COLORS = ['#58a6ff', '#3fb950', '#a371f7', '#f0883e', '#79c0ff']
print(f'Config ready: {len(MODELS)} models, {CORPUS_SIZE} docs, {len(GROUND_TRUTH)} queries')

Config ready: 5 models, 500 docs, 0 queries


In [20]:
print('Loading BEIR trec-covid corpus...')
dataset = load_dataset('BeIR/trec-covid', 'corpus')['corpus'].select(range(CORPUS_SIZE))
corpus_texts = [doc['text'] for doc in dataset]
print(f'Loaded {len(corpus_texts)} documents')
print(f'Sample doc: {corpus_texts[0][:150]}...')

# Generate ground truth: find top-10 similar docs for each query using e5-base-v2
print('\nGenerating ground truth (finding top-10 similar docs per query)...')
ref_model = SentenceTransformer('intfloat/e5-base-v2')
corpus_embs = ref_model.encode(['passage: ' + t for t in corpus_texts], 
                                batch_size=BATCH_SIZE, convert_to_numpy=True, 
                                normalize_embeddings=True, show_progress_bar=False)
index_ref = faiss.IndexFlatIP(corpus_embs.shape[1])
index_ref.add(corpus_embs.astype(np.float32))

for query in QUERIES:
    qe = ref_model.encode(['query: ' + query], normalize_embeddings=True)
    D, I = index_ref.search(qe.astype(np.float32), k=10)
    GROUND_TRUTH[query] = I[0].tolist()
    print(f"  '{query[:40]}...' → top-10 indices: {I[0].tolist()}")

del ref_model
print(f'Ground truth ready: {len(GROUND_TRUTH)} queries')

Loading BEIR trec-covid corpus...
Loaded 500 documents
Sample doc: OBJECTIVE: This retrospective chart review describes the epidemiology and clinical features of 40 patients with culture-proven Mycoplasma pneumoniae i...

Generating ground truth (finding top-10 similar docs per query)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  'How does COVID-19 affect lung tissue?...' → top-10 indices: [487, 181, 173, 138, 4, 444, 211, 439, 132, 314]
  'What are the symptoms of coronavirus inf...' → top-10 indices: [408, 487, 464, 388, 439, 16, 338, 488, 426, 173]
  'How is PCR testing used to detect COVID-...' → top-10 indices: [102, 15, 45, 338, 69, 227, 494, 439, 251, 247]
  'What treatments exist for severe COVID c...' → top-10 indices: [252, 249, 128, 338, 105, 354, 94, 439, 6, 225]
  'How does the spike protein enable viral ...' → top-10 indices: [357, 401, 409, 498, 378, 449, 347, 415, 159, 215]
Ground truth ready: 5 queries


In [21]:
def benchmark_model(model_name, model_id):
    print(f"\n{'='*60}\n  {model_name}\n{'='*60}")
    m = {'model': model_name}
    qpfx, dpfx = QUERY_PREFIX[model_name], DOC_PREFIX[model_name]

    # 1. Load
    t0 = time.perf_counter()
    model = SentenceTransformer(model_id)
    m['load_time_s'] = round(time.perf_counter() - t0, 2)
    print(f'  Load         : {m["load_time_s"]}s')

    # 2. Encode corpus -> throughput
    docs = [dpfx + t for t in corpus_texts]
    t0 = time.perf_counter()
    embs = model.encode(docs, batch_size=BATCH_SIZE, convert_to_numpy=True,
                        normalize_embeddings=True, show_progress_bar=False)
    enc_time = time.perf_counter() - t0
    m['encode_time_s']     = round(enc_time, 3)
    m['throughput_docs_s'] = round(len(corpus_texts) / enc_time, 1)
    m['embedding_dim']     = embs.shape[1]
    m['memory_mb']         = round(embs.nbytes / 1e6, 2)
    print(f'  Throughput   : {m["throughput_docs_s"]} docs/s  |  dim={m["embedding_dim"]}  |  {m["memory_mb"]} MB')

    # 3. FAISS index (IndexFlatIP = cosine for normalised vecs)
    index = faiss.IndexFlatIP(embs.shape[1])
    index.add(embs.astype(np.float32))

    # 4. Query latency — averaged over LATENCY_RUNS
    sample_q = qpfx + list(GROUND_TRUTH.keys())[0]
    lats = []
    for _ in range(LATENCY_RUNS):
        t0 = time.perf_counter()
        qe = model.encode([sample_q], normalize_embeddings=True)
        index.search(qe.astype(np.float32), k=TOP_K)
        lats.append((time.perf_counter() - t0) * 1000)
    m['latency_ms_mean'] = round(np.mean(lats), 2)
    m['latency_ms_p95']  = round(np.percentile(lats, 95), 2)
    m['latency_ms_p99']  = round(np.percentile(lats, 99), 2)
    print(f'  Latency      : mean={m["latency_ms_mean"]}ms  p95={m["latency_ms_p95"]}ms  p99={m["latency_ms_p99"]}ms')

    # 5. Recall@K and MRR
    recall_at = defaultdict(list)
    mrr_list, cosines = [], []
    for qtext, relevant in GROUND_TRUTH.items():
        qe = model.encode([qpfx + qtext], normalize_embeddings=True)
        D, I = index.search(qe.astype(np.float32), k=TOP_K)
        ret = I[0].tolist()
        cosines.extend(D[0].tolist())
        for k in [1, 3, 5]:
            hits = len(set(ret[:k]) & set(relevant))
            recall_at[k].append(hits / min(len(relevant), k))
        rr = next((1/r for r, d in enumerate(ret, 1) if d in relevant), 0.0)
        mrr_list.append(rr)
    m['recall@1'] = round(np.mean(recall_at[1]), 4)
    m['recall@3'] = round(np.mean(recall_at[3]), 4)
    m['recall@5'] = round(np.mean(recall_at[5]), 4)
    m['mrr']      = round(np.mean(mrr_list), 4)
    m['avg_top_cosine'] = round(np.mean(cosines), 4)
    m['cosine_scores']  = cosines
    print(f'  Recall@1/3/5 : {m["recall@1"]} / {m["recall@3"]} / {m["recall@5"]}')
    print(f'  MRR          : {m["mrr"]}  |  Avg cosine: {m["avg_top_cosine"]}')
    del model
    return m

all_metrics = [benchmark_model(name, mid) for name, mid in MODELS.items()]
df = pd.DataFrame([{k: v for k, v in m.items() if k != 'cosine_scores'} for m in all_metrics])
print('\nDone! Summary:')
print(df[['model','throughput_docs_s','latency_ms_mean','recall@5','mrr']].to_string(index=False))


  e5-base-v2


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  Load         : 6.62s
  Throughput   : 30.5 docs/s  |  dim=768  |  1.54 MB
  Latency      : mean=10.58ms  p95=12.16ms  p99=13.15ms
  Recall@1/3/5 : 1.0 / 1.0 / 1.0
  MRR          : 1.0  |  Avg cosine: 0.8308

  bge-base-en


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  Load         : 8.02s
  Throughput   : 25.5 docs/s  |  dim=768  |  1.54 MB
  Latency      : mean=8.61ms  p95=10.79ms  p99=12.19ms
  Recall@1/3/5 : 0.8 / 0.5333 / 0.56
  MRR          : 0.84  |  Avg cosine: 0.6298

  multilingual-e5-base


...

  Load         : 9.45s
  Throughput   : 27.2 docs/s  |  dim=768  |  1.54 MB
  Latency      : mean=14.99ms  p95=21.19ms  p99=25.15ms
  Recall@1/3/5 : 1.0 / 0.9333 / 0.76
  MRR          : 1.0  |  Avg cosine: 0.8398

  all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  Load         : 6.29s
  Throughput   : 276.6 docs/s  |  dim=384  |  0.77 MB
  Latency      : mean=5.0ms  p95=6.02ms  p99=6.69ms
  Recall@1/3/5 : 0.6 / 0.4667 / 0.52
  MRR          : 0.7  |  Avg cosine: 0.4388

  nomic-embed-v1


Loading weights:   0%|          | 0/112 [00:00<?, ?it/s]

  Load         : 7.74s
  Throughput   : 20.8 docs/s  |  dim=768  |  1.54 MB
  Latency      : mean=12.45ms  p95=15.31ms  p99=17.07ms
  Recall@1/3/5 : 0.8 / 0.8 / 0.6
  MRR          : 0.9  |  Avg cosine: 0.5624

Done! Summary:
               model  throughput_docs_s  latency_ms_mean  recall@5  mrr
          e5-base-v2               30.5            10.58      1.00 1.00
         bge-base-en               25.5             8.61      0.56 0.84
multilingual-e5-base               27.2            14.99      0.76 1.00
    all-MiniLM-L6-v2              276.6             5.00      0.52 0.70
      nomic-embed-v1               20.8            12.45      0.60 0.90


In [22]:
# ── 2-column Plotly Dashboard (2 charts per row) ────────────────────
model_names = df['model'].tolist()

from plotly.subplots import make_subplots
fig = make_subplots(
    rows=5, cols=2,
    subplot_titles=[
        'Throughput (docs/sec) ↑',
        'Mean Reciprocal Rank (MRR) ↑',
        'Query Latency mean/p95/p99 (ms) ↓',
        'Recall@1 / @3 / @5 ↑',
        'Embedding Dimension',
        'Cosine Score Distribution',
        'Index Memory (MB) ↓',
        'Model Load Time (s) ↓',
        'Avg Top-5 Cosine ↑',
        '',
    ],
    vertical_spacing=0.12, horizontal_spacing=0.15,
    specs=[[{}, {}], [{}, {}], [{}, {}], [{}, {}], [{}, {}]],
)

# R1C1: throughput
fig.add_trace(go.Bar(x=model_names, y=df['throughput_docs_s'], marker_color=COLORS,
    text=df['throughput_docs_s'].round(1), textposition='outside', showlegend=False), row=1, col=1)

# R1C2: MRR
fig.add_trace(go.Bar(x=model_names, y=df['mrr'], marker_color=COLORS,
    text=df['mrr'].round(3), textposition='outside', showlegend=False), row=1, col=2)

# R2C1: latency grouped (Mean, p95, p99)
for lbl, col, op in [('Mean','latency_ms_mean',1.0),('p95','latency_ms_p95',0.65),('p99','latency_ms_p99',0.4)]:
    fig.add_trace(go.Bar(x=model_names, y=df[col], name=lbl, marker_color=COLORS,
        opacity=op, text=df[col].round(1), textposition='outside', showlegend=(lbl=='Mean')), row=2, col=1)

# R2C2: recall grouped (R@1, R@3, R@5)
for k, col, op in [(1,'recall@1',1.0),(3,'recall@3',0.65),(5,'recall@5',0.4)]:
    fig.add_trace(go.Bar(x=model_names, y=df[col], name=f'R@{k}', marker_color=COLORS,
        opacity=op, text=df[col].round(3), textposition='outside', showlegend=(k==1)), row=2, col=2)

# R3C1: embedding dim
fig.add_trace(go.Bar(x=model_names, y=df['embedding_dim'], marker_color=COLORS,
    text=df['embedding_dim'], textposition='outside', showlegend=False), row=3, col=1)

# R3C2: violin cosine distribution
for i, m in enumerate(all_metrics):
    fig.add_trace(go.Violin(y=m['cosine_scores'], name=m['model'],
        box_visible=True, meanline_visible=True,
        line_color=COLORS[i], fillcolor=COLORS[i], opacity=0.55, showlegend=False), row=3, col=2)

# R4C1: memory
fig.add_trace(go.Bar(x=model_names, y=df['memory_mb'], marker_color=COLORS,
    text=df['memory_mb'].round(1), textposition='outside', showlegend=False), row=4, col=1)

# R4C2: load time
fig.add_trace(go.Bar(x=model_names, y=df['load_time_s'], marker_color=COLORS,
    text=df['load_time_s'].round(2), textposition='outside', showlegend=False), row=4, col=2)

# R5C1: avg cosine
fig.add_trace(go.Bar(x=model_names, y=df['avg_top_cosine'], marker_color=COLORS,
    text=df['avg_top_cosine'].round(3), textposition='outside', showlegend=False), row=5, col=1)

fig.update_layout(
    title=dict(text='<b>Embedding Model Comparison Dashboard</b><br>'
               '<sup>BEIR trec-covid · 500 docs · 5 queries · 10 latency runs</sup>',
               x=0.5, font_size=18),
    height=1600, barmode='group',
    paper_bgcolor='#0d1117', plot_bgcolor='#161b22',
    font=dict(color='#e6edf3', family='Inter,sans-serif', size=10),
    legend=dict(orientation='v', y=0.98, x=1.02,
                bgcolor='rgba(0,0,0,0)', bordercolor='#30363d'),
    margin=dict(t=130, b=40, l=60, r=120),
)

for r in range(1,6):
    for c in range(1,3):
        fig.update_xaxes(showgrid=False, tickfont_size=9, row=r, col=c)
        fig.update_yaxes(gridcolor='#21262d', gridwidth=0.5, row=r, col=c)
fig.show()

In [13]:
# ── Radar chart — overall profile per model ──────────────────────────
def minmax_norm(s, invert=False):
    lo, hi = s.min(), s.max()
    if hi == lo: return [0.5]*len(s)
    n = (s - lo)/(hi - lo)
    return (1-n).tolist() if invert else n.tolist()

axes = {
    'Throughput':  minmax_norm(df['throughput_docs_s']),
    'Low Latency': minmax_norm(df['latency_ms_mean'], invert=True),
    'Recall@5':    minmax_norm(df['recall@5']),
    'MRR':         minmax_norm(df['mrr']),
    'Cosine Qual': minmax_norm(df['avg_top_cosine']),
    'Low Memory':  minmax_norm(df['memory_mb'], invert=True),
}
cats = list(axes.keys()) + [list(axes.keys())[0]]

fig_r = go.Figure()
for i, mn in enumerate(model_names):
    vals = [axes[a][i] for a in list(axes.keys())] + [axes[list(axes.keys())[0]][i]]
    fig_r.add_trace(go.Scatterpolar(r=vals, theta=cats, fill='toself', name=mn,
        line_color=COLORS[i], fillcolor=COLORS[i], opacity=0.3))

fig_r.update_layout(
    title=dict(text='<b>Overall Model Profile</b><br><sup>All metrics normalised 0–1 (higher = better on every axis)</sup>', x=0.5),
    polar=dict(bgcolor='#161b22',
               radialaxis=dict(visible=True, range=[0,1], gridcolor='#30363d', tickfont_size=9),
               angularaxis=dict(gridcolor='#30363d')),
    paper_bgcolor='#0d1117', font=dict(color='#e6edf3'), height=520,
)
fig_r.show()

In [9]:
# ── Latency vs Recall@5 bubble chart — the sweet-spot view ───────────
fig_s = px.scatter(
    df, x='latency_ms_mean', y='recall@5', text='model',
    size='throughput_docs_s', color='model',
    color_discrete_sequence=COLORS, size_max=45,
    title='<b>Latency vs Recall@5</b>  (bubble = throughput docs/sec)<br>'
          '<sup>Ideal: bottom-right — low latency, high recall</sup>',
    labels={'latency_ms_mean': 'Query Latency mean (ms)', 'recall@5': 'Recall@5'},
    template='plotly_dark',
)
fig_s.update_traces(textposition='top center', textfont_size=11)
fig_s.update_layout(paper_bgcolor='#0d1117', plot_bgcolor='#161b22',
                    font=dict(color='#e6edf3'), height=460,
                    xaxis=dict(gridcolor='#21262d'), yaxis=dict(gridcolor='#21262d'))
fig_s.show()

In [14]:
# ── Summary table + winners ──────────────────────────────────────────
summary = df[['model','embedding_dim','throughput_docs_s','latency_ms_mean',
              'latency_ms_p95','recall@1','recall@3','recall@5','mrr',
              'avg_top_cosine','memory_mb','load_time_s']].copy()
summary.columns = ['Model','Dim','Thru(docs/s)','Lat.Mean(ms)','Lat.p95(ms)',
                   'R@1','R@3','R@5','MRR','AvgCos','Mem(MB)','Load(s)']
print('='*100)
print('EMBEDDING MODEL COMPARISON — FINAL SUMMARY')
print('='*100)
print(summary.to_string(index=False))
print('='*100)
print('\nWINNERS:')
for metric, col, better in [
    ('Throughput','throughput_docs_s','max'),
    ('Latency','latency_ms_mean','min'),
    ('Recall@5','recall@5','max'),
    ('MRR','mrr','max'),
    ('Cosine Qual','avg_top_cosine','max'),
    ('Low Memory','memory_mb','min'),
]:
    idx = df[col].idxmax() if better=='max' else df[col].idxmin()
    print(f'  {metric:<14} → {df.loc[idx,"model"]}  ({df.loc[idx,col]})')

EMBEDDING MODEL COMPARISON — FINAL SUMMARY
               Model  Dim  Thru(docs/s)  Lat.Mean(ms)  Lat.p95(ms)  R@1  R@3  R@5  MRR  AvgCos  Mem(MB)  Load(s)
          e5-base-v2  768          30.3         13.68        33.99  0.0  0.0  0.0  0.0  0.8308     1.54     6.31
         bge-base-en  768          33.5         10.99        24.82  0.0  0.0  0.0  0.0  0.6298     1.54     6.93
multilingual-e5-base  768          29.8         14.26        35.46  0.0  0.0  0.0  0.0  0.8398     1.54    10.25
    all-MiniLM-L6-v2  384         228.7          7.37        19.13  0.0  0.0  0.0  0.0  0.4388     0.77     6.15
      nomic-embed-v1  768          21.5         41.29       181.28  0.0  0.0  0.0  0.0  0.5624     1.54    35.80

WINNERS:
  Throughput     → all-MiniLM-L6-v2  (228.7)
  Latency        → all-MiniLM-L6-v2  (7.37)
  Recall@5       → e5-base-v2  (0.0)
  MRR            → e5-base-v2  (0.0)
  Cosine Qual    → multilingual-e5-base  (0.8398)
  Low Memory     → all-MiniLM-L6-v2  (0.77)
